In [1]:
import pandas as pd
import numpy as np

nav_df = pd.read_csv("D:/Programs/bluestock_mf_capstone/data/raw/02_nav_history.csv")

# Parsing dates to datetime objects
nav_df['date'] = pd.to_datetime(nav_df['date'])

# Sorting chronologically by fund, then date
nav_df = nav_df.sort_values(by=['amfi_code', 'date'])

nav_df = nav_df.drop_duplicates(subset=['amfi_code', 'date'])

# Validate NAV > 0 (A mutual fund cannot have a negative or zero price)
nav_df = nav_df[nav_df['nav'] > 0]

# Reindex to fill missing weekend dates, then forward-fill
# We group by amfi_code, create a complete date range from min to max date, and forward-fill the NAV
def fill_missing_dates(group):
    group = group.set_index('date')
    idx = pd.date_range(group.index.min(), group.index.max())
    group = group.reindex(idx)
    group['amfi_code'] = group['amfi_code'].ffill() # Fill the AMFI code down
    group['nav'] = group['nav'].ffill() # Forward-fill the price for weekends
    return group.reset_index().rename(columns={'index': 'date'})

print("Rows before weekend fill:", len(nav_df))
nav_df = nav_df.groupby('amfi_code').apply(fill_missing_dates).reset_index(drop=True)
print("Rows after weekend fill:", len(nav_df))

nav_df.to_csv("D:/Programs/bluestock_mf_capstone/data/processed/clean_nav.csv", index=False)

Rows before weekend fill: 46000
Rows after weekend fill: 64320


C:\Users\Shubham Singh\AppData\Local\Temp\ipykernel_39500\915003817.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nav_df = nav_df.groupby('amfi_code').apply(fill_missing_dates).reset_index(drop=True)


In [2]:
tx_df = pd.read_csv("D:/Programs/bluestock_mf_capstone/data/raw/08_investor_transactions.csv")

# Fixing date formats
tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'])

# Standardising transaction type to exactly: SIP, Lumpsum, Redemption
tx_df['transaction_type'] = tx_df['transaction_type'].str.strip().str.title().replace({'Sip': 'SIP'})

# Validating Amount > 0 (Cannot invest negative money)
tx_df = tx_df[tx_df['amount_inr'] > 0]

# Checking KYC Status Enum (Should only be 'Verified' or 'Pending')
valid_kyc = ['Verified', 'Pending']
invalid_kyc_count = len(tx_df[~tx_df['kyc_status'].isin(valid_kyc)])
print(f"Found {invalid_kyc_count} rows with invalid KYC status. Filtering them out.")
tx_df = tx_df[tx_df['kyc_status'].isin(valid_kyc)]

tx_df.to_csv("D:/Programs/bluestock_mf_capstone/data/processed/clean_transactions.csv", index=False)

Found 0 rows with invalid KYC status. Filtering them out.


In [3]:
perf_df = pd.read_csv("D:/Programs/bluestock_mf_capstone/data/raw/07_scheme_performance.csv")

# Ensuring numeric types for returns
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    perf_df[col] = pd.to_numeric(perf_df[col], errors='coerce')

# Flaging negative Sharpe ratios (Creates a new True/False column)
perf_df['negative_sharpe_flag'] = perf_df['sharpe_ratio'] < 0
negative_sharpe_count = perf_df['negative_sharpe_flag'].sum()
print(f"Flagged {negative_sharpe_count} funds with a negative Sharpe Ratio.")

# Validating Expense Ratio (0.1% to 2.5%)
initial_len = len(perf_df)
perf_df = perf_df[(perf_df['expense_ratio_pct'] >= 0.1) & (perf_df['expense_ratio_pct'] <= 2.5)]
print(f"Dropped {initial_len - len(perf_df)} funds due to invalid expense ratios.")

perf_df.to_csv("D:/Programs/bluestock_mf_capstone/data/processed/clean_performance.csv", index=False)

Flagged 0 funds with a negative Sharpe Ratio.
Dropped 0 funds due to invalid expense ratios.
